# 🧭 A* & Floyd-Warshall — Runnable Notebook

Companion to [`../tutorials/14_AStar_Floyd_Warshall.md`](../tutorials/14_AStar_Floyd_Warshall.md) and
[`../html/14_astar_floyd_warshall.html`](../html/14_astar_floyd_warshall.html).

**A\*** (heuristic-guided single goal) and **Floyd-Warshall** (all-pairs shortest paths).

## 1. A* — Dijkstra guided by a heuristic `f = g + h`

In [ ]:
import heapq

def dijkstra(adj, start):
    """Baseline for comparison (finalized-set Dijkstra, non-negative weights)."""
    dist = {start: 0}; done = set(); pq = [(0, start)]
    while pq:
        d, u = heapq.heappop(pq)
        if u in done: continue
        done.add(u)
        for v, w in adj[u]:
            if v not in done and d + w < dist.get(v, float("inf")):
                dist[v] = d + w; heapq.heappush(pq, (d + w, v))
    return dist

def a_star(adj, start, goal, h):
    """h(n) estimates the remaining distance to `goal`. Priority = g + h."""
    g = {start: 0}
    pq = [(h(start), start)]                       # ordered by f = g + h
    while pq:
        _f, u = heapq.heappop(pq)
        if u == goal:
            return g[u]                             # reached the goal, cheapest-first
        for v, w in adj[u]:
            ng = g[u] + w                            # actual cost to v through u
            if ng < g.get(v, float("inf")):
                g[v] = ng
                heapq.heappush(pq, (ng + h(v), v))   # priority uses the heuristic
    return float("inf")

adj = {0: [(1, 1), (2, 4)], 1: [(2, 1), (3, 5)], 2: [(3, 1)], 3: [(4, 2)], 4: []}
# admissible heuristic: never overestimates the true remaining distance to goal 4
H = {0: 4, 1: 3, 2: 2, 3: 1, 4: 0}
def h(n): return H[n]

best = a_star(adj, 0, 4, h)
print("A* shortest 0 -> 4:", best)
print("Dijkstra 0 -> 4   :", dijkstra(adj, 0)[4])
assert best == dijkstra(adj, 0)[4] == 5            # A* finds the optimal path

# with h = 0, A* is exactly Dijkstra
assert a_star(adj, 0, 4, lambda n: 0) == 5

## 2. Floyd-Warshall — all-pairs shortest paths in one triple loop

In [ ]:
def floyd_warshall(n, weight):
    """weight[i][j] = edge cost (INF if none, 0 on the diagonal). Returns all-pairs distances."""
    INF = float("inf")
    dist = [row[:] for row in weight]              # copy initial edge weights
    for k in range(n):                             # allow k as an intermediate...
        for i in range(n):                         # ...for every pair (i, j)
            for j in range(n):
                if dist[i][k] + dist[k][j] < dist[i][j]:
                    dist[i][j] = dist[i][k] + dist[k][j]
    return dist

INF = float("inf")
#            0    1    2    3
W = [[0,   3,   8, INF],       # 0->1 (3), 0->2 (8)
     [INF, 0,  -2, INF],       # 1->2 (-2)   (negative edge, no negative cycle)
     [INF, INF, 0,   2],       # 2->3 (2)
     [INF, INF, INF, 0]]
D = floyd_warshall(4, W)
print("0->2 (via 1 is cheaper):", D[0][2])       # min(8, 3 + (-2)) = 1
print("0->3                   :", D[0][3])        # 0->1->2->3 = 3 - 2 + 2 = 3
print("1->3                   :", D[1][3])        # 1->2->3 = -2 + 2 = 0
assert D[0][2] == 1 and D[0][3] == 3 and D[1][3] == 0

## 3. Detecting a negative cycle
A negative value on the **diagonal** means a vertex can cheapen a path back to itself.

In [ ]:
# 0->1 (1), 1->2 (-1), 2->0 (-1): the loop sums to -1
Wn = [[0,   1, INF],
      [INF, 0,  -1],
      [-1, INF,  0]]
Dn = floyd_warshall(3, Wn)
has_neg_cycle = any(Dn[i][i] < 0 for i in range(3))
print("diagonal:", [Dn[i][i] for i in range(3)])
print("negative cycle?", has_neg_cycle)
assert has_neg_cycle is True

## ✅ Recap
- **A\***: priority `f = g + h`; an **admissible** heuristic (never overestimates) keeps it optimal. `h = 0` ⇒ Dijkstra.
- **Floyd-Warshall**: `dist[i][j] = min(dist[i][j], dist[i][k] + dist[k][j])`, **`k` outermost**; `O(V³)`, all pairs.
- Floyd-Warshall handles **negative edges**; a **negative diagonal** flags a negative cycle.
- Pick by need: single-source non-neg → Dijkstra; source→goal + heuristic → A\*; all-pairs → Floyd-Warshall.

Next: [`15_Strongly_Connected_Components`](../tutorials/15_Strongly_Connected_Components.md).